In [62]:
import pdfplumber
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans
import json

In [ ]:
path_cartella = "/percorso/cartella_sentenze"
cartella = Path(path_cartella)
lista_pronuncie = list(cartella.glob('*.pdf'))

In [64]:
def primi_campi(file_path):

    sentenza_json = {'ID':'ID',
                 'numero': 'numero',
                 'anno': '2022',
                 'metadati': {'materia': 'materia',
                              'tipo_pronuncia': 'tipo_pronuncia',
                              'sezione': 'sezione',
                              'presidente': 'presidente',
                              'relatore': 'relatore',
                              'data_pubblicazione': 'data_pubblicazione'
                              },
                  'testo_completo': '', 
                  'testo': {},
                  'margini_sx':{}
                  }

    n_ = file_path.name.split('@')[-2][1:]
    sentenza_json['numero'] = n_
    sentenza_json['ID'] = n_ + '/' + '2022'

    return sentenza_json

In [65]:
def metadati_righe_margini(file_path, sentenza_json):

    with pdfplumber.open(file_path) as pdf:

        pdf_pagine = pdf.pages

        r1, r2, r3, r4 = pdf_pagine[0].extract_text_lines()[:4]

        for pagina in pdf_pagine:
            n_pagina = pagina.page_number
            pagina_tagliata = pagina.crop((0,0, 545, 790))
            righe = pagina_tagliata.extract_text_lines()
            lista_testi_righe = [riga['text'] for riga in righe]
            sentenza_json['testo'][f'pagina_{n_pagina}'] = lista_testi_righe
            lista_margini_sx = [riga['x0'] for riga in righe]
            sentenza_json['margini_sx'][f'pagina_{n_pagina}'] = lista_margini_sx

        condizioni = [r1['text'].split()[0]== 'Civile',r2['text'].split()[0]== 'Presidente:',r3['text'].split()[0]=='Relatore:',r4['text'].split()[0]== 'Data']

        if all(condizioni):

            metadati = sentenza_json['testo']['pagina_1'][:4]

            pagina_1_senza_box = sentenza_json['testo']['pagina_1'][4:]

            prima_riga = metadati[0].split()
            sentenza_json['metadati']['materia'] = prima_riga[0]
            sentenza_json['metadati']['sezione'] = prima_riga[3]
            sentenza_json['metadati']['tipo_pronuncia'] = prima_riga[1]

            seconda_riga = metadati[1]
            sentenza_json['metadati']['presidente'] = seconda_riga[12:]

            terza_riga = metadati[2]
            sentenza_json['metadati']['relatore'] = terza_riga[10:]

            quarta_riga = metadati[3].split()
            sentenza_json['metadati']['data_pubblicazione'] = quarta_riga[-1]

            sentenza_json['testo']['pagina_1'] = pagina_1_senza_box

            margini_1 = sentenza_json['margini_sx']['pagina_1']
            sentenza_json['margini_sx']['pagina_1'] = margini_1[4:]

        else:

            indici_righe_metadati = []

            pagina_1_senza_box = []
            
            for indice, riga in enumerate(sentenza_json['testo']['pagina_1']):

                if riga.split()[0] == 'Civile' and riga.split()[2] == 'Sez':
                    sentenza_json['metadati']['materia'] = riga.split()[0]
                    sentenza_json['metadati']['sezione'] = riga.split()[3]
                    sentenza_json['metadati']['tipo_pronuncia'] = riga.split()[1]
                    indici_righe_metadati.append(indice)
                
                elif riga.split()[0] == 'Presidente:':
                    sentenza_json['metadati']['presidente'] = riga[12:]
                    indici_righe_metadati.append(indice)

                elif riga.split()[0] == 'Relatore:':
                    sentenza_json['metadati']['relatore'] = riga[10:]
                    indici_righe_metadati.append(indice)

                elif riga.split()[0] == 'Data' and riga.split()[1] == 'pubblicazione:':
                    sentenza_json['metadati']['data_pubblicazione'] = riga.split()[-1]
                    indici_righe_metadati.append(indice)
                
                else:
                    pagina_1_senza_box.append(riga)
            
            sentenza_json['testo']['pagina_1'] = pagina_1_senza_box

            for ind in sorted(indici_righe_metadati, reverse=True):
                
                del sentenza_json['margini_sx']['pagina_1'][ind]

    
    #    try:       
     #   
      #      for pagina, lista_righe in sentenza_json['testo'].items():
       #         target_num_pagina = sentenza_json['testo'][pagina][-8]
        #        target_margine_sx = sentenza_json['margini_sx'][pagina][-8]
         #       margini = sentenza_json['margini_sx'][pagina]
          #      if 250 < target_margine_sx < 350 and any(x.isdigit() for x in target_num_pagina):
           #         sentenza_json['testo'][pagina] = lista_righe[:-8]
            #        sentenza_json['margini_sx'][pagina] = margini[:-8]
             #   else:
              #      sentenza_json['testo'][pagina] = lista_righe[:-7]
               #     sentenza_json['margini_sx'][pagina] = margini[:-7]
        
       # except:
            

           # print(file_path.name)
           # return None

            

    
    return sentenza_json


In [66]:
def rientri_testocompleto(sentenza_json):

    try:
        
        
        distribuzione = [margine for lista_margini in sentenza_json['margini_sx'].values() for margine in lista_margini]
        km_glob = KMeans(n_clusters = 2, random_state = 37)
        array_glob = np.array(distribuzione).reshape(-1,1)
        km_glob.fit_predict(array_glob)
        array_01_sxdx = km_glob.labels_
        chi_sx = np.argmin(km_glob.cluster_centers_)

        distribuzione_sx = array_glob[array_01_sxdx == chi_sx]  
        km_loc = KMeans(n_clusters = 2, random_state=37)
        array_loc = np.array(distribuzione_sx).reshape(-1,1)
        km_loc.fit_predict(array_loc)
        array_01_sxr = km_loc.labels_
        quale_sx = np.argmin(km_loc.cluster_centers_)

        prima_riga = np.where(array_01_sxdx==chi_sx, 0, 1)

        if abs(km_loc.cluster_centers_[0, 0] - km_loc.cluster_centers_[1, 0]) >= 15:

            sentenza_json['rientri'] = 'SI'

            seconda_riga0 = np.where(array_01_sxr==quale_sx, 0, 1)

            seconda_riga1 = prima_riga.copy()
            seconda_riga1[prima_riga==0] = seconda_riga0 

            smistamento = np.empty((2, len(distribuzione)), dtype = 'int')
            smistamento[0] = prima_riga
            smistamento[1] = seconda_riga1

            trasposta = smistamento.T

            contatore = 0
            for pagina, lista_righe in sentenza_json['testo'].items():
                for ind, riga in enumerate(lista_righe):
                    if (trasposta[ind+contatore] == [0,0]).all():
                        sentenza_json['testo_completo'] += ' ' + riga
                    elif (trasposta[ind+contatore] == [0,1]).all():
                        sentenza_json['testo_completo'] +='\n'+riga
                    elif (trasposta[ind+contatore] == [1,1]).all():
                        sentenza_json['testo_completo'] += '\n'+'___'+riga+'___'+'\n'
                contatore += len(lista_righe)

        else:
            sentenza_json['rientri'] = 'NO'
            trasposta = prima_riga.T
            contatore = 0
            for pagina, lista_righe in sentenza_json['testo'].items():
                for ind, riga in enumerate(lista_righe):
                    if trasposta[ind+contatore] == 0:
                        sentenza_json['testo_completo'] += ' ' + riga
                    elif trasposta[ind+contatore] == 1:
                        sentenza_json['testo_completo'] += '\n'+'___'+riga+'___'+'\n'
                contatore += len(lista_righe)

    except:
        return None


    return sentenza_json

In [ ]:
database_sentenze = []

contatore_2 = 0

for documento in lista_pronuncie:
    step1 = primi_campi(documento)
    step2 = metadati_righe_margini(documento, step1)
    if step2 is None:
        print(contatore_2)
        contatore_2 += 1
        continue
    step3 = rientri_testocompleto(step2)

    database_sentenze.append(step3)

path_database = "/percorso/database_sentenze.json"

with open(path_database, 'w', encoding='utf-8') as j:
    json.dump(database_sentenze, j, ensure_ascii=False, indent=4)
        

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
